# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN[:10])

hf_EWPVCGn


In [7]:
!pip install duckdb datasets pyarrow -q

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
# Unit of analysis: One row represents one content page for one client on one report date
# Time window: March 2026 (month= '2026-03')
unit_of_analysis = (
    "One row represents one content page for one client on one report date."
)

time_window = "March 2026 (month='2026-03')"

print(unit_of_analysis)
print(time_window)

One row represents one content page for one client on one report date.
March 2026 (month='2026-03')


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# Features:
    # impressions_prev_30d
    # clicks_prev_30d
    # sessions_prev_30d
    # avg_position
    # content_age_days

# Label:
    # trend_direction or is_declining_label

# Context:
    # content_id
    # client_id
    # content_type

# Excluded
    # trend_pct
    # trend_direction
    # provider_used
    #  model_used

# the reason: `trend_direction` defines the target and would cause target leakage

contract = {
    "features": [
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d",
        "avg_position",
        "content_age_days",
    ],
    "label": "trend_direction",
    "context": [
        "content_id",
        "client_id",
        "content_type",
    ],
    "excluded": [
        "trend_pct",
        "provider_used",
        "model_used",
    ]
}

contract

{'features': ['impressions_prev_30d',
  'clicks_prev_30d',
  'sessions_prev_30d',
  'avg_position',
  'content_age_days'],
 'label': 'trend_direction',
 'context': ['content_id', 'client_id', 'content_type'],
 'excluded': ['trend_pct', 'provider_used', 'model_used']}

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
import duckdb

con = duckdb.connect()

print("DuckDB is ready!")

DuckDB is ready!


In [10]:
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [ ]:
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [12]:
REL = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      9841378 │
└──────────────┘

In [14]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

### Query 1 — Verify the Grain


--> Each row represents one content item for one client on one report date



In [15]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

### Query 2 — Row Count + Date Span

--> Only rows where ga4_data_available IS TRUE are included in this analysis.

In [16]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

### Query 3 — Availability

--> 413,966 rows remain after filtering with ga4_data_available IS TRUE.

In [17]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# Rank content pages by refresh priority using historical search and engagement signals.
# ### Data limits

# - This analysis only covers the March 2026 partition.
# - Client history is unbalanced, so different clients have different amounts of historical data.
# - Rows where `ga4_data_available` is not `IS TRUE` were excluded.
# - This slice cannot describe future performance or causal effects.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.